# Phase 2 (L-NER) - S1: Model D, full fine-tuning, 5-fold

Model D's encoder (reads text in both directions) + a tagging head, all weights fine-tuned. Five models are trained, each on 4/5 of the 70 training documents and tested on the other fifth, so every document gets an honest score. The five models then vote on the test set.

**Before you run**
1. Accelerator: **GPU** (one T4 is enough). Internet not needed.
2. Attach three datasets: the Phase 2 data (fold1/fold2 text + gold, labels.txt, fold3-text.json), `legallm-tokenizers`, and your Model D checkpoint (`D_step10585_tokens650342400`).
3. **First run a quick check:** in the first code cell set `FOLDS_TO_RUN=[0]`, run everything, and look at the dev F1 after each epoch. Then set it back to `[0, 1, 2, 3, 4]` and run again; finished folds are skipped.
4. Results are saved to `/kaggle/working/p2_runs/S1_modelD_full/` (`submission_tuned.zip`, `submission_raw.zip`, `results.pkl`). Download `results.pkl` - the ensemble step needs it.

**Reading the log:** each line shows the fold, epoch, training loss and the strict macro-F1 on the held-out fifth. If the F1 is still rising at the last epoch, more epochs will help; if it peaks early and falls, fewer epochs will.


In [ ]:
# ---- Setup: paths, config, device ----
import os, sys, json, re, time, math, random, collections, zipfile, shutil, pickle
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

SMOKE = os.environ.get("P2_SMOKE", "0") == "1"      # tiny fast test (used only for my own checks)
CFG = dict(
    RUN_NAME="S1_modelD_full",
    WINDOW=512, STRIDE=256,            # tokens per window / step between windows
    EPOCHS=8, BATCH=8, ACCUM=2,        # effective batch = 16 windows
    LR_BASE=5e-5, LR_HEAD=1e-3, WD=0.01, WARMUP=0.06, DROPOUT=0.1,
    CLASS_WEIGHTS=True,                # heavier loss on entity tags (macro-F1 cares about rare labels)
    FOLDS_TO_RUN=[0, 1, 2, 3, 4],      # run [0] first as a quick check, then all five
    SEED=0,
)
if SMOKE:
    CFG.update(WINDOW=64, STRIDE=32, EPOCHS=2, BATCH=4, ACCUM=1, FOLDS_TO_RUN=[0, 1])

KIN = Path(os.environ.get("P2_INPUT_ROOT", "/kaggle/input"))
WORK = Path(os.environ.get("P2_WORK", "/kaggle/working"))
RUN_DIR = WORK / "p2_runs" / CFG["RUN_NAME"]; RUN_DIR.mkdir(parents=True, exist_ok=True)

def find_file(name):
    hits = sorted(KIN.rglob(name))
    if not hits: raise FileNotFoundError(f"'{name}' not found under {KIN}. Attach the dataset that contains it.")
    return hits[0]
DATA_DIR = find_file("fold1-text.json").parent
TOK_PATH = Path(os.environ.get("P2_TOK", "")) if os.environ.get("P2_TOK") else find_file("t2_unigram.model")
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("data:", DATA_DIR, "| tokenizer:", TOK_PATH, "| device:", DEVICE, "| run:", CFG["RUN_NAME"], "| SMOKE" if SMOKE else "")


In [ ]:
%%writefile p2_model_d.py
"""legal_d.py -- Model D: encoder-decoder legal language model + its trainer.

Model D (T5 / RedLLM-style encoder-decoder, tokenizer T2):
  * Encoder: bidirectional self-attention over the prompt.
  * Decoder: causal self-attention over the continuation + cross-attention
    to the encoder output.
  * Shared (tied) embedding for encoder input, decoder input and output head.
  * RMSNorm (pre-norm), RoPE in self-attention, QK-norm, GQA, SwiGLU, no biases.

Trainer: one process per GPU (DDP via torch.distributed.run), Muon optimizer
for the hidden weight matrices + AdamW for embeddings/norms, fp16 autocast
with GradScaler (T4 has no bf16), time-based warmup-stable-decay LR
schedule (so the LR always finishes its decay when the time budget ends),
flat checkpoints with resume, periodic push to a Kaggle dataset.

Run training:   python -m torch.distributed.run --standalone --nproc_per_node=2 legal_d.py d_config.json
Import helpers: import legal_d  (model, generation, scoring)
"""
import contextlib
import datetime
import itertools
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter
from pathlib import Path

import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint

ARCH_ID = "D"

MODEL_D_CONFIG = dict(
    vocab_size=32_000,        # T2 (SentencePiece Unigram), same as C
    d_model=768,
    n_enc_layer=12,
    n_dec_layer=12,
    n_head=12,
    n_kv_head=4,              # GQA 3:1 (self- and cross-attention)
    d_head=64,
    d_ff=2048,                # SwiGLU (same FFN parameter count as C's 4x GELU)
    max_enc_len=1024,
    max_dec_len=1024,
    qk_norm=True,
    mlp="swiglu",
    rope_theta=10_000.0,
    tokenizer_id="T2",
    arch="encoder-decoder",
    tied_embeddings=True,
)

# ---------------------------------------------------------------------------
# Tokenizer (T2)
# ---------------------------------------------------------------------------
sp = None
EOS_ID = None
DEC_START_ID = None
BANNED_IN_OUTPUT = []


def load_tokenizer(path):
    global sp, EOS_ID, DEC_START_ID, BANNED_IN_OUTPUT
    import sentencepiece as spm
    sp = spm.SentencePieceProcessor()
    sp.load(str(path))
    EOS_ID = sp.eos_id()
    assert EOS_ID >= 0, "T2 tokenizer has no <eos>"
    DEC_START_ID = sp.bos_id() if sp.bos_id() >= 0 else EOS_ID      # decoder start token
    BANNED_IN_OUTPUT = sorted({sp.pad_id(), sp.unk_id(), sp.bos_id(), sp.eos_id()} - {-1})
    return sp


def encode(text):
    return sp.encode(text, out_type=int)


def decode(ids):
    return sp.decode(ids)


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x32 = x.float()
        return (x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight.float()).to(x.dtype)


def rope_cache(n, d_head, theta):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2).float() / d_head))
    freqs = torch.outer(torch.arange(n).float(), inv_freq)
    return freqs.cos(), freqs.sin()                      # [n, d_head/2]


def apply_rope(x, cos, sin):
    """x [B, H, T, d]; cos/sin [T, d/2]. Rotate-half form, computed in fp32."""
    cos, sin = cos[None, None].float(), sin[None, None].float()
    half = x.shape[-1] // 2
    x1, x2 = x[..., :half].float(), x[..., half:].float()
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1).to(x.dtype)


def expand_kv(t, n_rep):
    if n_rep == 1:
        return t
    B, H, T, D = t.shape
    return t[:, :, None].expand(B, H, n_rep, T, D).reshape(B, H * n_rep, T, D)


class KVCache:
    """Decoder self-attention cache (allocated on first use)."""

    def __init__(self, max_len):
        self.max_len, self.pos, self.k, self.v = max_len, 0, None, None

    def update(self, k, v):
        if self.k is None:
            B, H, _, D = k.shape
            self.k = torch.zeros(B, H, self.max_len, D, device=k.device, dtype=k.dtype)
            self.v = torch.zeros_like(self.k)
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


class SelfAttention(nn.Module):
    def __init__(self, cfg, causal):
        super().__init__()
        d, h, kv, dh = cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"]
        self.causal, self.n_head, self.n_kv_head, self.d_head = causal, h, kv, dh
        self.q_proj = nn.Linear(d, h * dh, bias=False)
        self.k_proj = nn.Linear(d, kv * dh, bias=False)
        self.v_proj = nn.Linear(d, kv * dh, bias=False)
        self.o_proj = nn.Linear(h * dh, d, bias=False)
        self.q_norm = RMSNorm(dh) if cfg["qk_norm"] else None
        self.k_norm = RMSNorm(dh) if cfg["qk_norm"] else None

    def forward(self, x, cos, sin, key_mask=None, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        first_chunk = cache is None or cache.pos == 0
        if cache is not None:
            assert T == 1 or cache.pos == 0, "cached decoding feeds one token at a time"
            k, v = cache.update(k, v)
        rep = self.n_head // self.n_kv_head
        k, v = expand_kv(k, rep), expand_kv(v, rep)
        if key_mask is not None:                                  # [B, Tk] True = real token
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask[:, None, None, :])
        else:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal and T > 1 and first_chunk)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head))


class CrossAttention(nn.Module):
    """Decoder queries attend to the encoder output (no RoPE: T5-style)."""

    def __init__(self, cfg):
        super().__init__()
        d, h, kv, dh = cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"]
        self.n_head, self.n_kv_head, self.d_head = h, kv, dh
        self.q_proj = nn.Linear(d, h * dh, bias=False)
        self.k_proj = nn.Linear(d, kv * dh, bias=False)
        self.v_proj = nn.Linear(d, kv * dh, bias=False)
        self.o_proj = nn.Linear(h * dh, d, bias=False)
        self.q_norm = RMSNorm(dh) if cfg["qk_norm"] else None
        self.k_norm = RMSNorm(dh) if cfg["qk_norm"] else None

    def project_kv(self, enc_out):
        B, S, _ = enc_out.shape
        k = self.k_proj(enc_out).view(B, S, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(enc_out).view(B, S, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.k_norm is not None:
            k = self.k_norm(k)
        return k, v

    def forward(self, x, enc_kv, enc_mask=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q = self.q_norm(q)
        rep = self.n_head // self.n_kv_head
        k, v = expand_kv(enc_kv[0], rep), expand_kv(enc_kv[1], rep)
        mask = enc_mask[:, None, None, :] if enc_mask is not None else None
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, f = cfg["d_model"], cfg["d_ff"]
        self.gate_proj = nn.Linear(d, f, bias=False)
        self.up_proj = nn.Linear(d, f, bias=False)
        self.down_proj = nn.Linear(f, d, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg["d_model"])
        self.attn = SelfAttention(cfg, causal=False)
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = SwiGLU(cfg)

    def forward(self, x, cos, sin, key_mask=None):
        x = x + self.attn(self.attn_norm(x), cos, sin, key_mask)
        return x + self.mlp(self.mlp_norm(x))


class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.self_norm = RMSNorm(cfg["d_model"])
        self.self_attn = SelfAttention(cfg, causal=True)
        self.cross_norm = RMSNorm(cfg["d_model"])
        self.cross_attn = CrossAttention(cfg)
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = SwiGLU(cfg)

    def forward(self, x, cos, sin, enc_out=None, enc_mask=None, enc_kv=None, cache=None):
        if enc_kv is None:
            enc_kv = self.cross_attn.project_kv(enc_out)
        x = x + self.self_attn(self.self_norm(x), cos, sin, cache=cache)
        x = x + self.cross_attn(self.cross_norm(x), enc_kv, enc_mask)
        return x + self.mlp(self.mlp_norm(x))


def _ce_chunk(h, w, t):
    logits = F.linear(h, w).float()
    lse = torch.logsumexp(logits, dim=-1)
    valid = (t != -100).float()
    nll = (lse - logits.gather(-1, t.clamp(min=0)[:, None]).squeeze(-1)) * valid
    return torch.stack([nll.sum(), (lse.pow(2) * valid).sum()])


def chunked_loss(h, w, targets, chunk=4096):
    """[1, 3] = [sum of token NLL, number of scored tokens, sum of logsumexp^2].
    Logits are built chunk by chunk (and recomputed in backward), so the full
    [tokens x 32k] logit tensor is never stored."""
    h = h.reshape(-1, h.shape[-1])
    t = targets.reshape(-1)
    parts = []
    for s in range(0, h.shape[0], chunk):
        hs, ts = h[s:s + chunk], t[s:s + chunk]
        if torch.is_grad_enabled() and h.requires_grad:
            parts.append(grad_checkpoint(_ce_chunk, hs, w, ts, use_reentrant=False))
        else:
            parts.append(_ce_chunk(hs, w, ts))
    tot = torch.stack(parts).sum(0)
    count = (t != -100).sum().to(tot.dtype)
    return torch.stack([tot[0], count, tot[1]]).unsqueeze(0)


class EncoderDecoderLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        d, V = config["d_model"], config["vocab_size"]
        self.embed = nn.Embedding(V, d)
        self.enc_blocks = nn.ModuleList([EncoderBlock(config) for _ in range(config["n_enc_layer"])])
        self.enc_norm = RMSNorm(d)
        self.dec_blocks = nn.ModuleList([DecoderBlock(config) for _ in range(config["n_dec_layer"])])
        self.dec_norm = RMSNorm(d)
        self.head = nn.Linear(d, V, bias=False)
        self.head.weight = self.embed.weight                  # tied
        cos, sin = rope_cache(max(config["max_enc_len"], config["max_dec_len"]), config["d_head"], config["rope_theta"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.grad_ckpt = False
        self.amp = True                                        # fp16 autocast on GPU (fp32 weights)
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
        for name, p in self.named_parameters():                 # scaled residual-output init
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                n_res = 2 * config["n_enc_layer"] if name.startswith("enc_") else 3 * config["n_dec_layer"]
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(n_res))

    def _run(self, blk, *args, **kwargs):
        if self.grad_ckpt and self.training and torch.is_grad_enabled():
            return grad_checkpoint(blk, *args, use_reentrant=False, **kwargs)
        return blk(*args, **kwargs)

    def encode(self, input_ids, attention_mask=None):
        """Contract method: encoder hidden states [B, T, d]. attention_mask
        (1 = real token) marks padding; positions start at 0 in every row
        (pad on the right)."""
        T = input_ids.shape[1]
        x = self.embed(input_ids)
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        key_mask = attention_mask.bool() if attention_mask is not None else None
        for blk in self.enc_blocks:
            x = self._run(blk, x, cos, sin, key_mask)
        return self.enc_norm(x)

    def decode_hidden(self, dec_ids, enc_out, enc_mask=None):
        T = dec_ids.shape[1]
        x = self.embed(dec_ids)
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        m = enc_mask.bool() if enc_mask is not None else None
        for blk in self.dec_blocks:
            x = self._run(blk, x, cos, sin, enc_out=enc_out, enc_mask=m)
        return self.dec_norm(x)

    def forward(self, enc_ids, dec_ids, targets=None, enc_mask=None):
        """No targets -> logits [B, Q, V]. With targets -> [1, 3] sums
        (NLL, count, lse^2); targets = -100 are ignored."""
        use_amp = enc_ids.is_cuda and self.embed.weight.dtype == torch.float32 and self.amp
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            enc_out = self.encode(enc_ids, enc_mask)
            h = self.decode_hidden(dec_ids, enc_out, enc_mask)
            if targets is None:
                return self.head(h)
            return chunked_loss(h, self.head.weight, targets)

    def decoder_step(self, tok, pos, caches, cross_kvs, enc_mask):
        x = self.embed(tok)
        cos, sin = self.rope_cos[pos:pos + tok.shape[1]], self.rope_sin[pos:pos + tok.shape[1]]
        for blk, cache, kv in zip(self.dec_blocks, caches, cross_kvs):
            x = blk(x, cos, sin, enc_mask=enc_mask, enc_kv=kv, cache=cache)
        return self.head(self.dec_norm(x))

    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy", **kw):
        """Contract method (one prompt, greedy): returns prompt + continuation ids."""
        assert kv_cache and sampler == "greedy"
        toks = generate_batch(self, [input_ids[0].tolist()], max_new_tokens=max_new_tokens, **kw)[0]
        return torch.tensor([input_ids[0].tolist() + toks], device=input_ids.device)


def build_model(config):
    """Contract function."""
    return EncoderDecoderLM(config)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


# ---------------------------------------------------------------------------
# Generation


def step_of(path):
    return int(Path(path).stem.split("_step")[1].split("_tokens")[0])


def find_latest_checkpoint(prefix, search_dirs, extract_to):
    cands = []
    for d in search_dirs:
        d = Path(d)
        if d.exists():
            cands += [p for p in d.rglob(f"{prefix}_step*.pt")]
    best_zip = None
    for d in search_dirs:
        d = Path(d)
        if not d.exists():
            continue
        for zpath in d.rglob("*.zip"):
            try:
                with zipfile.ZipFile(zpath) as zf:
                    for name in zf.namelist():
                        base = Path(name).name
                        if base.startswith(f"{prefix}_step") and base.endswith(".pt"):
                            if best_zip is None or step_of(base) > best_zip[0]:
                                best_zip = (step_of(base), zpath, name)
            except zipfile.BadZipFile:
                continue
    best_file = max(cands, key=step_of) if cands else None
    if best_zip is not None and (best_file is None or best_zip[0] > step_of(best_file)):
        _, zpath, name = best_zip
        dest = Path(extract_to) / Path(name).name
        with zipfile.ZipFile(zpath) as zf, zf.open(name) as src, open(dest, "wb") as dst:
            shutil.copyfileobj(src, dst)
        return dest
    return best_file




def load_for_inference(path, device, half=False):
    """fp32 weights + fp16 autocast on GPU (same numerics as training)."""
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    model = build_model(ckpt["config"])
    model.load_state_dict(ckpt["model_state"])
    meta = {k: ckpt.get(k) for k in ("step", "tokens_seen", "config", "run_id")}
    del ckpt
    model = model.to(device).eval()
    if half and device.type == "cuda":
        model = model.half()
        model.rope_cos, model.rope_sin = model.rope_cos.float(), model.rope_sin.float()
    return model, meta


In [ ]:
# ---- Data, word splitter, tags, Viterbi decoding, strict F1 (same logic as the step-1 offset test) ----
def norm_spans(g):
    out = []
    for x in g:
        if isinstance(x, dict):
            s = x.get("start", x.get("start_char")); e = x.get("end", x.get("end_char")); l = x.get("label", x.get("type"))
        else: s, e, l = x[0], x[1], x[2]
        out.append((int(s), int(e), str(l)))
    return out
def load_fold(fold):
    txt = json.load(open(DATA_DIR / f"fold{fold}-text.json", encoding="utf-8"))
    gold = json.load(open(DATA_DIR / f"fold{fold}-gold.json", encoding="utf-8"))
    if isinstance(txt, list): txt = {str(d.get("id")): d.get("text") for d in txt}
    if isinstance(gold, list): gold = {str(d.get("id")): d.get("entities", d.get("spans", [])) for d in gold}
    return {str(k): v for k, v in txt.items()}, {str(k): norm_spans(v) for k, v in gold.items()}
texts, golds = {}, {}
for f in (1, 2):
    t, g = load_fold(f); texts.update(t); golds.update(g)
test_texts = json.load(open(DATA_DIR / "fold3-text.json", encoding="utf-8"))
if isinstance(test_texts, list): test_texts = {str(d.get("id")): d.get("text") for d in test_texts}
test_texts = {str(k): v for k, v in test_texts.items()}
LABELS = [l.strip() for l in open(DATA_DIR / "labels.txt", encoding="utf-8") if l.strip()]
TAGS = ["O"] + [f"{p}-{l}" for l in LABELS for p in "BI"]
T2I = {t: i for i, t in enumerate(TAGS)}; NT = len(TAGS)
ALL_IDS = sorted(texts)
random.seed(0); _sh = ALL_IDS[:]; random.shuffle(_sh)            # identical to step 1 -> identical folds
K = 5; FOLDS = [sorted(_sh[k::K]) for k in range(K)]
print(f"train docs {len(ALL_IDS)}, test docs {len(test_texts)}, labels {len(LABELS)}, folds {[len(f) for f in FOLDS]}")
print("fold first ids:", [f[:2] for f in FOLDS])

PRE = re.compile(r"[^\W\d_]+|\d+|_|[^\w\s]", re.UNICODE)
def pretok(text): return [(m.start(), m.end()) for m in PRE.finditer(text)]

def spans_to_tags(toks, spans):
    tags = [0] * len(toks)
    starts = {s: i for i, (s, e) in enumerate(toks)}; ends = {e: i for i, (s, e) in enumerate(toks)}
    for s, e, l in sorted(spans):
        if s in starts and e in ends and starts[s] <= ends[e]:
            a, b = starts[s], ends[e]
            if any(t != 0 for t in tags[a:b + 1]): continue
            tags[a] = T2I["B-" + l]
            for k in range(a + 1, b + 1): tags[k] = T2I["I-" + l]
    return tags

def tags_to_spans(toks, path):
    out, cur = [], None
    for (s, e), t in zip(toks, path):
        name = TAGS[t]
        if name.startswith("B-") or (name.startswith("I-") and (cur is None or cur[2] != name[2:])):
            if cur: out.append(tuple(cur))
            cur = [s, e, name[2:]]
        elif name.startswith("I-"): cur[1] = e
        else:
            if cur: out.append(tuple(cur)); cur = None
    if cur: out.append(tuple(cur))
    return out

# allowed tag transitions: I-X may only follow B-X or I-X
TRANS_OK = np.ones((NT, NT), bool)
for j, tj in enumerate(TAGS):
    if tj.startswith("I-"):
        TRANS_OK[:, j] = False
        TRANS_OK[T2I["B-" + tj[2:]], j] = TRANS_OK[j, j] = True
START_OK = np.array([not t.startswith("I-") for t in TAGS])
NEG = -1e9
TRANS_PEN = np.where(TRANS_OK, 0.0, NEG)

def viterbi(logp):
    N = len(logp)
    if N == 0: return []
    score = np.where(START_OK, logp[0], NEG); back = np.zeros((N, NT), np.int16)
    for t in range(1, N):
        cand = score[:, None] + TRANS_PEN
        back[t] = cand.argmax(0); score = cand.max(0) + logp[t]
    path = [int(score.argmax())]
    for t in range(N - 1, 0, -1): path.append(int(back[t][path[-1]]))
    return path[::-1]

def strict_f1(gold, pred, labels=LABELS):
    per = {}
    for l in labels:
        tp = fp = fn = 0
        for d in set(gold) | set(pred):
            g = {x for x in gold.get(d, []) if x[2] == l}; p = {tuple(x[:3]) for x in pred.get(d, []) if x[2] == l}
            tp += len(g & p); fp += len(p - g); fn += len(g - p)
        pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
        per[l] = dict(p=pr, r=rc, f1=2 * pr * rc / (pr + rc) if pr + rc else 0.0, support=tp + fn)
    used = [l for l in labels if per[l]["support"] > 0] or labels
    m = lambda k: sum(per[l][k] for l in used) / len(used)
    return m("f1"), m("p"), m("r"), per


In [ ]:
# ---- Tokenize each word piece with Model D's tokenizer (T2) and cut documents into windows ----
import sentencepiece as spm
sp = spm.SentencePieceProcessor(); sp.load(str(TOK_PATH))
PAD_ID = sp.pad_id() if sp.pad_id() >= 0 else 0
UNK_ID = sp.unk_id()
_cache = {}
def piece_ids(piece, after_space):
    key = (piece, after_space)
    if key in _cache: return _cache[key]
    ids = sp.encode(piece, out_type=int)                 # SentencePiece adds a leading "▁" (word-start marker)
    if not after_space and ids:                          # attached piece (punctuation, footnote digit): drop the marker
        p0 = sp.id_to_piece(ids[0])
        if p0 == "▁": ids = ids[1:]
        elif p0.startswith("▁"):
            alt = sp.piece_to_id(p0[1:])
            if alt != UNK_ID: ids = [alt] + ids[1:]
    if not ids: ids = [UNK_ID]
    _cache[key] = ids
    return ids

class Doc:
    def __init__(self, did, text, spans=None):
        self.id, self.text = did, text
        self.toks = pretok(text)
        ids, first = [], []
        for s, e in self.toks:
            first.append(len(ids)); ids.extend(piece_ids(text[s:e], s == 0 or text[s - 1].isspace()))
        self.ids = np.array(ids, dtype=np.int64); self.first = np.array(first, dtype=np.int64)
        self.n = len(self.ids)
        self.pos2tok = np.full(self.n, -1, dtype=np.int64); self.pos2tok[self.first] = np.arange(len(self.toks))
        self.tags = None
        if spans is not None:
            self.tags = np.array(spans_to_tags(self.toks, spans), dtype=np.int64)
            self.lab = np.full(self.n, -100, dtype=np.int64); self.lab[self.first] = self.tags
            self.gold = spans

def window_starts(n, W, stride, offset=0):
    st = {0, max(0, n - W)} | set(range(offset, max(1, n - W + 1), stride))
    return sorted(st)

t0 = time.time()
DOCS = {i: Doc(i, texts[i], golds[i]) for i in ALL_IDS}
TEST_DOCS = {i: Doc(i, test_texts[i]) for i in sorted(test_texts)}
ntok = sum(d.n for d in DOCS.values()); npiece = sum(len(d.toks) for d in DOCS.values())
print(f"tokenized in {time.time()-t0:.0f}s: {npiece} word pieces -> {ntok} tokens ({ntok/npiece:.2f} tokens/piece); "
      f"test: {sum(d.n for d in TEST_DOCS.values())} tokens")
print("tokens per doc: median", int(np.median([d.n for d in DOCS.values()])), "max", max(d.n for d in DOCS.values()))
print("windows per epoch (all train docs):", sum(len(window_starts(d.n, CFG['WINDOW'], CFG['STRIDE'])) for d in DOCS.values()))


In [ ]:
# ---- Load Model D, keep only its encoder, add a tagging head; training and prediction code ----
sys.path.insert(0, str(Path.cwd()))
import p2_model_d as legal_d
CKPT_PATH = Path(os.environ["P2_CKPT"]) if os.environ.get("P2_CKPT") else legal_d.find_latest_checkpoint("D", [KIN], WORK)
assert CKPT_PATH is not None, "No D_step*.pt checkpoint found. Attach the Model D checkpoint dataset."
_ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
D_CONFIG, D_STATE = _ck["config"], _ck["model_state"]
print(f"Model D checkpoint: {CKPT_PATH.name}  step={_ck.get('step')}  tokens={_ck.get('tokens_seen')}")
del _ck

class NERModel(nn.Module):
    def __init__(self, dropout):
        super().__init__()
        self.base = legal_d.build_model(D_CONFIG); self.base.load_state_dict(D_STATE)
        del self.base.dec_blocks, self.base.dec_norm                    # the decoder is not used for tagging
        self.drop = nn.Dropout(dropout); self.head = nn.Linear(D_CONFIG["d_model"], NT)
        nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)
    def forward(self, ids, mask):
        with torch.autocast("cuda", dtype=torch.float16, enabled=ids.is_cuda):
            h = self.base.encode(ids, mask)
        return self.head(self.drop(h.float()))

def batchify(arrs, pad):
    L = max(len(a) for a in arrs); out = np.full((len(arrs), L), pad, dtype=np.int64); m = np.zeros((len(arrs), L), dtype=np.int64)
    for i, a in enumerate(arrs): out[i, :len(a)] = a; m[i, :len(a)] = 1
    return torch.from_numpy(out), torch.from_numpy(m)

@torch.no_grad()
def predict_probs(model, docs, bs=16):
    """-> {doc_id: [n_pieces, NT] probabilities}, averaged over overlapping windows (centre of a window counts more)."""
    model.eval(); W, S = CFG["WINDOW"], CFG["STRIDE"]; res = {}
    for d in docs:
        acc = np.zeros((len(d.toks), NT), np.float32); wsum = np.zeros(len(d.toks), np.float32)
        starts = window_starts(d.n, W, S)
        for b in range(0, len(starts), bs):
            chunk = starts[b:b + bs]
            ids, mask = batchify([d.ids[a:a + W] for a in chunk], PAD_ID)
            pr = torch.softmax(model(ids.to(DEVICE), mask.to(DEVICE)), -1).cpu().numpy()
            for k, a in enumerate(chunk):
                L = min(W, d.n - a); ti = d.pos2tok[a:a + L]; keep = ti >= 0
                if not keep.any(): continue
                pos = np.nonzero(keep)[0]; w = 1.0 - np.abs(pos - (L - 1) / 2) / (L / 2 + 1)    # triangular weight
                acc[ti[keep]] += pr[k, pos] * w[:, None]; wsum[ti[keep]] += w
        res[d.id] = acc / np.maximum(wsum, 1e-6)[:, None]
    return res

def decode(probs, docs, bias=None):
    """probs -> spans, with constrained Viterbi. bias: optional {label: log-bonus} added to that label's B/I tags."""
    out = {}
    b = np.zeros(NT, np.float32)
    if bias:
        for l, v in bias.items(): b[T2I["B-" + l]] = b[T2I["I-" + l]] = v
    for d in docs:
        lp = np.log(np.clip(probs[d.id], 1e-8, 1.0)) + b
        out[d.id] = tags_to_spans(d.toks, viterbi(lp))
    return out

def dev_score(model, dev_docs):
    pr = predict_probs(model, dev_docs); sp_ = decode(pr, dev_docs)
    return strict_f1({d.id: d.gold for d in dev_docs}, sp_), pr

def tag_weights(train_docs):
    cnt = np.bincount(np.concatenate([d.tags for d in train_docs]), minlength=NT).astype(np.float64) + 1
    w = np.clip((cnt[0] / cnt) ** 0.25, 1.0, 6.0); w[0] = 1.0
    return torch.tensor(w if CFG["CLASS_WEIGHTS"] else np.ones(NT), dtype=torch.float32)

def train_fold(train_docs, dev_docs, fold):
    seed = CFG["SEED"] + fold; random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = NERModel(CFG["DROPOUT"]).to(DEVICE)
    W, S, BS, AC = CFG["WINDOW"], CFG["STRIDE"], CFG["BATCH"], CFG["ACCUM"]
    n_win = sum(len(window_starts(d.n, W, S)) for d in train_docs)
    total = CFG["EPOCHS"] * math.ceil(n_win / (BS * AC)); warm = max(1, int(CFG["WARMUP"] * total))
    decay = [p for n, p in model.named_parameters() if p.ndim >= 2]; nodecay = [p for n, p in model.named_parameters() if p.ndim < 2]
    base_ids = {id(p) for p in model.base.parameters()}
    groups = []
    for plist, wd in ((decay, CFG["WD"]), (nodecay, 0.0)):
        groups.append(dict(params=[p for p in plist if id(p) in base_ids], lr=CFG["LR_BASE"], weight_decay=wd))
        groups.append(dict(params=[p for p in plist if id(p) not in base_ids], lr=CFG["LR_HEAD"], weight_decay=wd))
    groups = [g for g in groups if g["params"]]
    opt = torch.optim.AdamW(groups); base_lrs = [g["lr"] for g in opt.param_groups]
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")
    wts = tag_weights(train_docs).to(DEVICE); step = 0; history = []
    print(f"[fold {fold}] train docs {len(train_docs)}, dev docs {len(dev_docs)}, windows/epoch {n_win}, updates {total}")
    for ep in range(CFG["EPOCHS"]):
        model.train(); t0 = time.time()
        items = []                                         # new random window phase each epoch = free augmentation
        for d in train_docs:
            for a in window_starts(d.n, W, S, offset=random.randrange(S)): items.append((d, a))
        random.shuffle(items); run, nrun = 0.0, 0
        for b in range(0, len(items), BS):
            chunk = items[b:b + BS]
            ids, mask = batchify([d.ids[a:a + W] for d, a in chunk], PAD_ID)
            lab, _ = batchify([d.lab[a:a + W] for d, a in chunk], -100)
            lab = torch.where(mask.bool(), lab, torch.full_like(lab, -100))
            logits = model(ids.to(DEVICE), mask.to(DEVICE))
            loss = F.cross_entropy(logits.view(-1, NT), lab.to(DEVICE).view(-1), weight=wts, ignore_index=-100)
            scaler.scale(loss / AC).backward(); run += loss.item(); nrun += 1
            if (b // BS + 1) % AC == 0 or b + BS >= len(items):
                f = (step + 1) / warm if step < warm else 0.5 * (1 + math.cos(math.pi * min(1.0, (step - warm) / max(1, total - warm))))
                for g, lr0 in zip(opt.param_groups, base_lrs): g["lr"] = lr0 * f
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
        (f1, pr_, rc_, per), _ = dev_score(model, dev_docs)
        history.append(dict(epoch=ep + 1, loss=run / max(nrun, 1), dev_macro_f1=f1))
        print(f"[fold {fold}] epoch {ep+1}/{CFG['EPOCHS']}  loss {run/max(nrun,1):.4f}  dev strict macro-F1 {f1:.4f} (P {pr_:.3f} R {rc_:.3f})  {time.time()-t0:.0f}s", flush=True)
    return model, history


In [ ]:
# ---- Run the folds, collect out-of-fold predictions, predict the test set, write submission files ----
oof, test_probs, hist = {}, [], {}
done_path = RUN_DIR / "results.pkl"
if done_path.exists():                                   # resume after a closed tab / restarted session
    _r = pickle.load(open(done_path, "rb")); oof, test_probs, hist = _r["oof"], _r["test_probs"], _r["hist"]
    print("resuming; folds already done:", sorted(hist))
test_docs_list = [TEST_DOCS[i] for i in sorted(TEST_DOCS)]
for fold in CFG["FOLDS_TO_RUN"]:
    if fold in hist: print(f"fold {fold} already done, skipping"); continue
    dev_ids = set(FOLDS[fold]); tr = [DOCS[i] for i in ALL_IDS if i not in dev_ids]; dv = [DOCS[i] for i in FOLDS[fold]]
    model, h = train_fold(tr, dv, fold)
    oof.update(predict_probs(model, dv)); test_probs.append(predict_probs(model, test_docs_list)); hist[fold] = h
    pickle.dump(dict(oof=oof, test_probs=test_probs, hist=hist, tags=TAGS, cfg=CFG), open(done_path, "wb"))
    del model; torch.cuda.empty_cache()

# ---- results ----
docs_oof = [DOCS[i] for i in ALL_IDS if i in oof]
spans = decode(oof, docs_oof)
f1, p, r, per = strict_f1({d.id: d.gold for d in docs_oof}, spans)
print(f"\n=== OUT-OF-FOLD (each doc predicted by a model that never saw it): {len(docs_oof)} docs ===")
print(f"strict macro-F1 {f1:.4f}   precision {p:.4f}   recall {r:.4f}")
for l in LABELS: print(f"   {l:10s} F1 {per[l]['f1']:.3f}  P {per[l]['p']:.3f}  R {per[l]['r']:.3f}  n={per[l]['support']}")

# per-label bias tuned on the out-of-fold predictions (raises macro-F1 directly; coarse grid to avoid overfitting)
GRID = [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0]
def score_bias(b): return strict_f1({d.id: d.gold for d in docs_oof}, decode(oof, docs_oof, b))[0]
bias = {l: 0.0 for l in LABELS}; best = f1
for _ in range(2):
    for l in LABELS:
        for v in GRID:
            trial = dict(bias, **{l: v}); s = score_bias(trial)
            if s > best + 1e-6: best, bias = s, trial
print(f"\nwith per-label bias tuned on the folds: macro-F1 {best:.4f} (was {f1:.4f});  bias = { {k: v for k, v in bias.items() if v} }")

# ---- test predictions: average the fold models ----
def write_submission(name, b):
    n = len(test_probs)
    avg = {d.id: sum(tp[d.id] for tp in test_probs) / n for d in test_docs_list}
    sp_ = decode(avg, test_docs_list, b); pred = {}
    for d in test_docs_list:
        pred[d.id] = [[int(s), int(e), l] for s, e, l in sp_[d.id] if 0 <= s < e <= len(d.text) and l in LABELS]
    assert set(pred) == set(test_texts), "test ids mismatch"
    out = RUN_DIR / name; out.mkdir(exist_ok=True)
    json.dump(pred, open(out / "prediction.json", "w"))
    with zipfile.ZipFile(RUN_DIR / f"submission_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z: z.write(out / "prediction.json", "prediction.json")
    cnt = collections.Counter(x[2] for v in pred.values() for x in v)
    print(f"{name}: {sum(len(v) for v in pred.values())} entities in {len(pred)} docs  {dict(cnt)}")
    return avg
if test_probs:
    print(f"\nTest predictions from {len(test_probs)} fold model(s):")
    write_submission("raw", None); write_submission("tuned", bias)
    print(f"\nUpload this file to Codabench: {RUN_DIR}/submission_tuned.zip  (or submission_raw.zip)")
    print("Also keep results.pkl - the later ensemble step uses it.")
